# MCP Concepts 05: Lifecycle, Sampling & Roots, Security, and Production Notes

## Problem card

- **What this notebook is:** the wrap-up. Notebooks 01-04 built and connected
  real servers; this one covers what a working demo doesn't force you to
  learn -- the full protocol lifecycle, two advanced primitives (sampling and
  roots) most tutorials skip, a security checklist grounded in what actually
  broke in this course (not generic advice), and what changes between a
  notebook demo and a production deployment.
- **Recap of what's real so far in this course:**

| Notebook | Server(s) | Transport | Primitives shown |
|---|---|---|---|
| 01 | `knowledge_ops_server.py` | stdio + HTTP | tools, resources (static + templated), prompts |
| 02 | `calculator_server.py` + `knowledge_ops_server.py` | stdio | tools, via a LangGraph agent |
| 03 | + `orders_server.py` (3 servers total) | stdio | tools, cross-server routing, a real name-collision bug |
| 04 | DeepWiki (public, third-party) | Streamable HTTP | tools, a real context-overflow bug |
| 05 (this one) | `sampling_demo_server.py` | stdio | sampling (real), roots (conceptual) |

## Protocol lifecycle, in full

Every notebook so far called `session.initialize()` without dwelling on what
it actually does. In full:

```mermaid
sequenceDiagram
    participant C as Client
    participant S as Server

    C->>S: initialize(protocolVersion, capabilities, clientInfo)
    S-->>C: InitializeResult(protocolVersion, capabilities, serverInfo)
    C->>S: notifications/initialized
    Note over C,S: Connection is now live -- list_tools/call_tool/etc. all valid from here
    C->>S: list_tools / list_resources / list_prompts
    S-->>C: current capabilities
    Note over S: If the server's tools change later (e.g. a plugin loads)...
    S-->>C: notifications/tools/list_changed
    Note over C: ...the client should re-fetch list_tools rather than assume it's stale-safe
```

**Capability negotiation** is the point of `initialize`/`InitializeResult`:
client and server each declare what they support (sampling, roots, specific
protocol features) *before* anything else happens, so neither side assumes a
capability the other doesn't actually have.

In [1]:
import os
import sys
import warnings
from dotenv import load_dotenv, find_dotenv

warnings.filterwarnings("ignore")
load_dotenv(find_dotenv(usecwd=True))
sys.path.insert(0, os.getcwd())
import shared

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from mcp.types import CreateMessageResult, TextContent
from openai import OpenAI

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("Setup complete.")

Setup complete.


## Sampling: the server asks the CLIENT's LLM for a completion

Every server in this course so far either did no LLM work at all (01) or
had *us* (the client-side agent) call the LLM and pass the server's tool
results back in. **Sampling inverts that**: the server itself can ask the
client to run an LLM completion on its behalf, via `ctx.sample(...)` --
useful when a server wants to do LLM-powered work (like summarizing) without
holding its own API key or paying for its own model calls; the *client*
pays and the *client* controls which model actually runs.

```mermaid
sequenceDiagram
    participant C as Client (has the API key)
    participant S as Server (sampling_demo_server.py, has NO API key)

    C->>S: call_tool("summarize_via_client_llm", {text})
    S->>C: sampling/createMessage request (via ctx.sample)
    Note over C: Client's sampling_callback runs a REAL OpenAI call
    C-->>S: CreateMessageResult (the generated text)
    S-->>C: tool result (the server just relays what the client generated)
```

The server file (`servers/sampling_demo_server.py`) has no `OPENAI_API_KEY`
reference anywhere -- it cannot call an LLM on its own. Below, the *client*
provides a `sampling_callback` that makes the real call.

In [2]:
async def sampling_callback(context, params):
    """Runs on the CLIENT side. The server asked for a completion; we decide
    how to actually generate it -- here, a real OpenAI call."""
    user_text = params.messages[-1].content.text
    response = openai_client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        messages=[{"role": "user", "content": user_text}],
        max_tokens=params.maxTokens or 100,
    )
    text = response.choices[0].message.content
    return CreateMessageResult(
        role="assistant",
        content=TextContent(type="text", text=text),
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        stopReason="endTurn",
    )


async def run_sampling_demo():
    params = StdioServerParameters(command="python3", args=["servers/sampling_demo_server.py"])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write, sampling_callback=sampling_callback) as session:
            await session.initialize()
            result = await session.call_tool("summarize_via_client_llm", {
                "text": (
                    "LangGraph is a low-level orchestration framework for building "
                    "stateful, multi-actor applications with LLMs, providing durable "
                    "execution, human-in-the-loop support, and memory management."
                )
            })
            print("Server's tool returned (generated by the CLIENT's LLM, not the server's):")
            print(result.content[0].text)

await run_sampling_demo()

Server's tool returned (generated by the CLIENT's LLM, not the server's):
LangGraph is a framework designed for creating stateful, multi-actor applications with LLMs, featuring durable execution, human-in-the-loop capabilities, and memory management.


## Roots: the client tells the server what it may access (conceptual)

Roots run in the opposite direction of sampling: the **client** tells the
**server** which resources (typically filesystem paths or URIs) it's
allowed to work within, via a `list_roots_callback` the client registers --
the server can call `list_roots` to discover its own boundaries, rather than
the client having to trust the server to self-limit.

```mermaid
sequenceDiagram
    participant C as Client
    participant S as Server (e.g. a filesystem-access MCP server)

    Note over C: Client decides: "this server may only touch ./project/data"
    S->>C: roots/list request
    C-->>S: [{"uri": "file:///project/data"}]
    Note over S: Server should now scope its own file operations to that root
```

This course doesn't ship a live roots example -- none of our servers touch a
real filesystem -- but the mechanism matters most for exactly that case: a
real filesystem or file-browsing MCP server should always be handed an
explicit root boundary by the client, not be trusted to infer a safe scope
on its own.

## Security checklist -- grounded in what actually happened in this course

Not generic advice -- every item below maps to something this course either
demonstrated or directly risked:

1. **Treat every tool result as untrusted input**, even from a friendly
   server. Notebook 04's `ask_question` result was fed straight back into
   the LLM's context -- a malicious server could return text crafted to look
   like new instructions ("ignore previous instructions and...").
2. **Duplicate tool names across servers are a real, silent failure mode**,
   not a contrived edge case -- notebook 03 reproduced this deterministically.
   Before combining MCP servers you don't fully control, check for name
   collisions explicitly; don't assume `get_tools()` would have warned you.
3. **A server's response size is not bounded by the protocol** -- notebook
   04 hit a genuine 167K-token context overflow from one tool call. Scope
   which tools you bind, and consider validating/truncating tool output
   size before it reaches the LLM in a production system.
4. **stdio vs. HTTP have different trust models.** A stdio server is code
   you chose to run as a subprocess -- audit it like any other dependency.
   An HTTP server is a live network endpoint -- apply the same scrutiny you'd
   give any third-party API (notebook 04's checklist applies in full).
5. **This ecosystem moves fast enough that APIs get removed, not just
   added.** `ctx.sample()` -- used above -- was removed entirely in FastMCP 4
   (this course pins `fastmcp==3.4.6` deliberately for that reason). Pin
   versions and re-verify against current docs before trusting older
   tutorials, including this one.
6. **No write tools exist anywhere in this course's servers.** Every tool
   across `calculator_server.py`, `knowledge_ops_server.py`, and
   `orders_server.py` is read-only by construction -- the same
   read-only-by-design boundary this repo's `text-to-sql-analytics` project
   uses for its own database tool. If you add a write-capable MCP tool,
   that tool needs explicit authorization/approval handling, not just a
   docstring warning.

## Production considerations

| Concern | What this course did (toy version) | What production needs |
|---|---|---|
| Process lifecycle | `stdio_client` spawns/tears down automatically; `shared.start_http_server`/`stop_server` for our one HTTP demo | Real process supervision (systemd, a container orchestrator, or a managed MCP host) with restart-on-crash |
| Failure handling | Errors surfaced as Python exceptions in a notebook cell | Explicit retries/timeouts around every `call_tool`, and a decision for what the *agent* does when a tool call fails (retry, ask the user, degrade gracefully) |
| Observability | Print statements | Tracing tool calls (this repo's `eval-and-observability` skill wires Phoenix for exactly this), so a slow or failing MCP call is visible, not silent |
| Auth | None -- every server here is either local or explicitly no-auth-public | Any real HTTP MCP server serving non-public data needs real authentication, not just network reachability |
| Scale | One agent, servers started fresh per notebook run | Connection pooling/reuse across requests instead of spawning a fresh stdio subprocess per call |

For a fuller, production-oriented MCP build (a runnable CLI, Qdrant-backed
retrieval, a multi-agent capstone, and its own security/testing notebooks),
see `teaching/mcp_all/` in this repo -- this course's five notebooks are the
concepts; that project is a worked end-to-end application built on them.

## Explain like I'm 12

Think back over this whole mini-course like learning to use a telephone,
step by step. Notebook 01 taught you the phone itself -- how to dial, how to
know if someone even picked up. Notebook 02 was making your first real call
to a friend and asking them to do something for you. Notebook 03 was
learning that if two of your friends have the *exact same name* saved in
your phone, calling that name might reach the wrong friend -- annoying, but
now you know to check before it matters. Notebook 04 was calling a stranger
you found in a public phone book -- still useful, but you're more careful
what you say and how much you believe. And this notebook is the "grown-up"
lesson: how the phone call actually gets set up in the first place, a weird
trick where your friend can ask *you* to think of the answer instead of
thinking of it themselves (sampling), and a checklist of things a careful
person always double-checks before making or answering any call.

## Checkpoint questions (spanning all 5 notebooks)

1. **Q: In notebook 01, what's the difference between a tool and a resource,
   and why does that distinction affect whether an LLM call happens?**
   A: A tool is invoked by model decision (`call_tool`, costs an LLM
   decision and can be called wrong); a resource is fetched deterministically
   by the client's own code via `read_resource`, with no LLM involved.

2. **Q: In notebook 02, why did `agent_node` need `ainvoke` instead of
   `invoke`?**
   A: MCP tools loaded via `langchain-mcp-adapters` are async by nature
   (real I/O to a subprocess), so the whole graph has to run asynchronously
   for `ToolNode` to be able to execute them.

3. **Q: What specifically caused notebook 03's `check_status` bug -- an LLM
   mistake, or something else?**
   A: Something else -- `ToolNode.tools_by_name` is a plain dict keyed by
   tool name, so two identically-named tools from different MCP servers
   collapse to one entry; the LLM's tool choice was actually correct.

4. **Q: What real failure did notebook 04 hit, and why couldn't
   `list_tools()` have warned about it in advance?**
   A: A ~167K-token context overflow from `read_wiki_contents`. `list_tools()`
   only returns names/descriptions/schemas -- it says nothing about how much
   data a tool's *result* might contain, which can only really be discovered
   by seeing (or reasoning about) an actual call.

5. **Q: In this notebook's sampling demo, which side actually held the
   `OPENAI_API_KEY` -- the client or the server?**
   A: The client. `sampling_demo_server.py` never references any API key;
   it asks the client to generate text via `ctx.sample()`, and the client's
   `sampling_callback` is what actually calls OpenAI.

6. **Q: Across the whole course, which transport was used for servers you
   wrote yourself, and which for a server you didn't control?**
   A: stdio for every self-written server (01-03, plus this notebook's
   sampling demo); Streamable HTTP for the one third-party server
   (notebook 04's DeepWiki) -- matching the rule of thumb from notebook 01's
   comparison table.